# Step2 — Integración espacial (Ecuador, 2015–2025)

**Proyecto:** HCI Ecuador (Hazard Chronicity Index)
**Fase:** 2 — Integración espacial (`hazard_event_geo` + `admin_boundaries`)
**Autor:** Jerson Achicanoy (FAO OER) · implementación asistida
**Fecha:** 2026-08-05 (v2: se añade confirmación cruzada por parroquia)

**Objetivo de este notebook:** convertir la tabla `hazard_event` (Fase 1) en
una capa geoespacial de puntos, cargar y estandarizar los límites
administrativos oficiales de Ecuador (provincia/cantón/parroquia, fuente
INEC vía HDX COD-AB), validar espacialmente la geolocalización reportada
(¿el punto realmente cae dentro del cantón que SNGRE dice?), **confirmar
casos ambiguos usando el nombre/código de parroquia (ADM3) reportado**,
armonizar nombres oficiales, y exportar ambas capas en GeoPackage/GeoJSON.

**No calcula** índices ni hotspots — eso es Fase 3 y Fase 9 respectivamente.

Notebook comentado línea por línea (regla del proyecto 2026-08-05). Lógica
inline; `src/utilidades` solo aporta helpers transversales.

Ver `docs/methodology/02_spatial_integration.md` y
`docs/executive/02_spatial_integration.md`.

**English:** Spatial Integration

Loads the official administrative boundaries, validates the province/canton codes of each event, builds the event point layer and cross-checks each event's canton by point-in-polygon and by parish.

Outputs: `hazard_event_geo.gpkg`, `admin_boundaries.gpkg` and simplified GeoJSON layers in `data/processed/`.

## 0. Configuración del entorno

**English:** 0. Setup

In [ ]:
# Fase 2 tampoco requiere GEE ni CDS: trabaja sobre shapefiles ya
# descargados manualmente (ver excepción de adquisición de datos en el
# documento técnico) y sobre la salida de la Fase 1.

# Standard library: system path handling
import sys  # para añadir la raíz del repo a sys.path
# Object-oriented file paths
from pathlib import Path  # rutas multiplataforma

# Project root is the parent of the notebooks folder
PROJECT_ROOT = Path.cwd().parent  # notebook vive en <repo>/notebooks/
# Make the project's src package importable
sys.path.insert(0, str(PROJECT_ROOT))

# Helpers to load the YAML config and resolve folder paths
from src.utilidades.config_utils import load_config, resolve_path  # config YAML
# Project logger factory
from src.utilidades.logging_utils import get_logger  # logging estándar
# Helper to create output folders
from src.utilidades.io_utils import ensure_dir  # helper de directorios

# Tabular data handling
import pandas as pd  # tablas
# Geospatial data handling
import geopandas as gpd  # datos geoespaciales (geometrías + atributos)
# Point geometry for event locations
from shapely.geometry import Point  # construir geometría punto desde lat/lon

# Show up to 30 columns when printing DataFrames
pd.set_option("display.max_columns", 30)  # inspección visual más cómoda

In [ ]:
# Path to the central configuration file
CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"  # ruta al YAML
# Load the configuration into a dictionary
config = load_config(CONFIG_PATH)  # config completa del proyecto

# Create a logger that writes to the logs folder
logger = get_logger("step2_spatial_integration", config, log_dir=PROJECT_ROOT / "logs")
logger.info("Fase 2 iniciada (v2, con confirmación por parroquia) — configuración cargada desde %s", CONFIG_PATH)

PROJECT_CRS = config["project"]["crs"]  # "EPSG:4326" — CRS de trabajo único del proyecto

## 1. Carga y estandarización de los límites administrativos

Fuente: HDX "Ecuador - Subnational Administrative Boundaries" (cod-ab-ecu),
declarada por HDX como derivada de INEC. Se validó empíricamente (fuera de
este notebook, ver documento técnico) que `ADMx_PCODE = "EC" + código DPA de
INEC`, y que ese código DPA coincide 100% con `province_code`/`canton_code`
de `hazard_event` (Fase 1). Se cargan ADM0 (país, solo contexto
cartográfico), ADM1 (provincia) y ADM2 (cantón); ADM3 (parroquia) se carga
también — ya no solo como capa opcional, sino como insumo activo de la
confirmación cruzada de la sección 4b. ADM4 (656 MB, sub-parroquial) NO se
carga — excede el alcance del esquema `Admin_Boundary` del diseño.

**English:** 1. Load and standardize administrative boundaries

In [ ]:
# Directorio y nombres de archivo de los shapefiles, desde config.yaml.
# Folder with the raw administrative boundary files
admin_dir = resolve_path(config, "raw_dir", base_dir=Path.cwd()) / "admin_boundaries"
# File names of each administrative level, from the config
admin_files = config["admin_boundaries"]["files"]  # dict: nivel -> nombre de archivo
pcode_prefix = config["admin_boundaries"]["pcode_prefix"]  # "EC"

# Log where the boundaries are loaded from
logger.info("Cargando límites administrativos desde %s", admin_dir)

# Leer cada nivel con geopandas (motor pyogrio: más rápido y liviano en memoria
# que el motor fiona por defecto, relevante dado que ADM2/ADM3 pesan 100-250 MB).
# Read the country boundary
adm0 = gpd.read_file(admin_dir / admin_files["adm0"], engine="pyogrio")  # país
# Read the province boundaries
adm1 = gpd.read_file(admin_dir / admin_files["adm1"], engine="pyogrio")  # provincia
# Read the canton boundaries
adm2 = gpd.read_file(admin_dir / admin_files["adm2"], engine="pyogrio")  # cantón
# Read the parish boundaries
adm3 = gpd.read_file(admin_dir / admin_files["adm3"], engine="pyogrio")  # parroquia

# Loop over all levels to report their size
for name, gdf in [("adm0", adm0), ("adm1", adm1), ("adm2", adm2), ("adm3", adm3)]:
    # Log the number of features and the CRS of each level
    logger.info("%s: %d features, CRS=%s", name, len(gdf), gdf.crs)

In [ ]:
# Reproyectar a PROJECT_CRS si hiciera falta (ya vienen en EPSG:4326, pero no
# se asume — se fuerza explícitamente para que el resto del pipeline no
# dependa de que la fuente no cambie de CRS en una actualización futura).
# Reproject the country layer to the project CRS
adm0 = adm0.to_crs(PROJECT_CRS)
# Reproject the province layer
adm1 = adm1.to_crs(PROJECT_CRS)
# Reproject the canton layer
adm2 = adm2.to_crs(PROJECT_CRS)
# Reproject the parish layer
adm3 = adm3.to_crs(PROJECT_CRS)

# Convert an official place code to the national DPA code
def pcode_to_dpa(series: pd.Series, prefix: str) -> pd.Series:
    """Convierte un Pcode de HDX (p. ej. 'EC0101') al código DPA de INEC ('0101').

    Args:
        series: columna de Pcodes (ADM1_PCODE, ADM2_PCODE, etc.).
        prefix: prefijo a eliminar (config.yaml -> admin_boundaries.pcode_prefix).

    Returns:
        Serie de códigos DPA, como texto (preserva ceros a la izquierda).

    English:
    Convert an official place code (for example 'EC09') to the national DPA code (for example '09').

    Args:
        series: Place codes with the country prefix.
        prefix: Country prefix to remove.

    Returns:
        The codes without the prefix.
    """
    return series.str.replace(prefix, "", regex=False)  # quitar el prefijo 'EC'

# Añadir el código DPA (formato exacto de hazard_event) a cada nivel.
adm1["province_code"] = pcode_to_dpa(adm1["ADM1_PCODE"], pcode_prefix)  # p. ej. '01'
adm2["canton_code"] = pcode_to_dpa(adm2["ADM2_PCODE"], pcode_prefix)  # p. ej. '0101'
adm3["parish_code"] = pcode_to_dpa(adm3["ADM3_PCODE"], pcode_prefix)  # p. ej. '010150'

# Renombrar columnas de nombre oficial a snake_case consistente con el
# esquema Hazard_Event (ver Fase 0 §7 y su extensión de nomenclatura).
# Standardize province column names
adm1 = adm1.rename(columns={"ADM1_ES": "province_name_official"})
# Standardize canton column names
adm2 = adm2.rename(columns={"ADM2_ES": "canton_name_official", "ADM1_ES": "province_name_official"})
# Standardize parish column names
adm3 = adm3.rename(columns={"ADM3_ES": "parish_name_official", "ADM2_ES": "canton_name_official"})

# El código de cantón "padre" de cada parroquia (viene directo de ADM2_PCODE
# dentro de la propia capa ADM3) — se usará en la sección 4b para confirmar
# a qué cantón pertenece cada evento según su parroquia reportada.
# Canton code that each parish belongs to
adm3["canton_code_from_parish"] = pcode_to_dpa(adm3["ADM2_PCODE"], pcode_prefix)

# Preview the first three cantons
adm2[["canton_code", "canton_name_official", "province_name_official", "geometry"]].head(3)

## 2. Validación del cruce de códigos (Admin_Boundary vs. Hazard_Event)

Antes de usar los límites administrativos como referencia oficial, se
verifica que sus códigos DPA coincidan con los ya presentes en
`hazard_event` — si no coincidieran, cualquier join posterior fallaría
silenciosamente (quedarían geometrías nulas sin que el pipeline lo detecte).

**English:** 2. Validate the code match between boundaries and events

In [ ]:
# Cargar la tabla de eventos ya limpia de la Fase 1.
# Folder with processed tables
processed_dir = resolve_path(config, "processed_dir", base_dir=Path.cwd())
# Load the hazard events produced by the ingestion notebook
hazard_event = pd.read_parquet(processed_dir / "hazard_event.parquet")
# Log the number of events loaded
logger.info("hazard_event cargado: %d filas", len(hazard_event))

# Asegurar que los códigos de hazard_event sean texto con ceros a la
# izquierda preservados (Parquet puede haberlos dejado como número/float).
# Zero-pad province codes to 2 digits
hazard_event["province_code"] = hazard_event["province_code"].astype(str).str.zfill(2)
# Zero-pad canton codes to 4 digits
hazard_event["canton_code"] = hazard_event["canton_code"].astype(str).str.zfill(4)
# Zero-pad parish codes to 6 digits
hazard_event["parish_code"] = hazard_event["parish_code"].astype(str).str.zfill(6)  # usado en 4b

# Comparar conjuntos de códigos únicos en ambas direcciones.
# Provinces present in the events
he_provinces = set(hazard_event["province_code"].unique())
# Provinces present in the official boundaries
adm1_provinces = set(adm1["province_code"].unique())
# Cantons present in the events
he_cantons = set(hazard_event["canton_code"].unique())
# Cantons present in the official boundaries
adm2_cantons = set(adm2["canton_code"].unique())

# Log the province coverage
logger.info(
    "Provincias — en hazard_event sin match en ADM1: %d | en ADM1 sin match en hazard_event: %d",
    len(he_provinces - adm1_provinces), len(adm1_provinces - he_provinces),
)
# Log the canton coverage
logger.info(
    "Cantones — en hazard_event sin match en ADM2: %d | en ADM2 sin match en hazard_event: %d",
    len(he_cantons - adm2_cantons), len(adm2_cantons - he_cantons),
)

# Estas dos aserciones son la validación formal de la Fase 2: si fallan, no
# se continúa (mejor detenerse aquí que producir un GeoPackage con joins
# silenciosamente incompletos).
# Check every event province exists in the boundaries
assert he_provinces <= adm1_provinces, "Hay provincias en hazard_event sin límite administrativo"
# Check every event canton exists in the boundaries
assert he_cantons <= adm2_cantons, "Hay cantones en hazard_event sin límite administrativo"
# Log that the code match is complete
logger.info("Validación de cruce de códigos: OK (100%% de cobertura provincia y cantón)")

## 3. Construcción de la capa de puntos `hazard_event_geo`

Se construye la geometría de cada evento a partir de `latitude`/`longitude`
(ya validadas en la Fase 1 — `coord_valid`). Se asigna `PROJECT_CRS`
(EPSG:4326), consistente con los límites administrativos.

**English:** 3. Build the event point layer

In [ ]:
# Point(x, y) = Point(longitud, latitud) — shapely usa orden (x, y), no (lat, lon).
# Build a point for every event from its longitude and latitude
geometry = [Point(xy) for xy in zip(hazard_event["longitude"], hazard_event["latitude"])]

# Construir el GeoDataFrame de eventos, mismo CRS que los límites administrativos.
# Create the GeoDataFrame of events
hazard_event_geo = gpd.GeoDataFrame(hazard_event, geometry=geometry, crs=PROJECT_CRS)

# Log the number of points
logger.info("hazard_event_geo construido: %d puntos", len(hazard_event_geo))
# Preview the first two rows
hazard_event_geo.head(2)

## 4. Validación espacial punto-en-polígono

Control de calidad geoespacial: ¿el punto (lat/lon) reportado por SNGRE cae
realmente dentro del polígono del cantón que SNGRE también reportó
(`canton_code`)? Un desajuste indica error de geocodificación (coordenada
mal capturada) o de codificación administrativa (cantón mal asignado) en la
fuente — información valiosa para decidir qué registros usar con más o
menos confianza en fases posteriores.

**English:** 4. Point-in-polygon validation

In [ ]:
# sjoin espacial: para cada punto, encontrar en qué polígono de adm2 cae
# ('within' = el punto está estrictamente dentro del polígono).
# Keep only the canton columns needed for the join
adm2_for_join = adm2[["canton_code", "canton_name_official", "province_name_official", "geometry"]]
# Spatial join: find the canton polygon that contains each point
joined = gpd.sjoin(
    hazard_event_geo,
    adm2_for_join.rename(columns={"canton_code": "canton_code_geo", "canton_name_official": "canton_name_geo"}),
    how="left",  # conservar todos los eventos aunque no caigan en ningún polígono (p. ej. coordenada en el mar)
    predicate="within",
)

# Si un punto cae exactamente sobre un borde compartido por dos cantones,
# sjoin puede duplicar la fila (un match por polígono). Se deduplica
# conservando la primera coincidencia — no afecta el conteo total de eventos.
# Row count before removing duplicates
n_before = len(joined)
# Remove duplicates caused by points on shared borders
joined = joined[~joined.index.duplicated(keep="first")]
# Log the number of duplicated rows removed
logger.info("Filas duplicadas por sjoin en bordes compartidos: %d", n_before - len(joined))

# Bandera de validación: ¿el cantón geométrico coincide con el reportado?
# Flag events whose reported canton equals the geometric canton
joined["canton_geo_matches_reported"] = joined["canton_code_geo"] == joined["canton_code"]

# Share of events where both agree
match_rate = joined["canton_geo_matches_reported"].mean()
# Count events that fall in no polygon
n_no_polygon = joined["canton_code_geo"].isna().sum()  # punto no cayó en ningún cantón (p. ej. mar, error grave)
# Log the agreement rate and the unmatched count
logger.info(
    "Validación espacial: %.1f%% de coincidencia cantón reportado vs. geométrico | %d puntos sin polígono",
    match_rate * 100, n_no_polygon,
)

In [ ]:
# Desglose de discrepancias por provincia — ayuda a detectar si el problema
# se concentra en alguna zona (p. ej. cantones costeros pequeños, más
# propensos a error de redondeo de coordenadas).
# Events where reported and geometric cantons differ
mismatches = joined.loc[~joined["canton_geo_matches_reported"] & joined["canton_code_geo"].notna()]
# Log the number of disagreements
logger.info("Eventos con cantón reportado distinto del geométrico: %d de %d", len(mismatches), len(joined))
# Show the provinces with most disagreements
mismatches["province"].value_counts().head(10)

## 4b. Confirmación cruzada por parroquia (ADM3) — resuelve la ambigüedad de la sección 4

**Origen de esta sección:** Jerson notó que, en los eventos con problemas de
coordenada, SNGRE de todos modos reportó un nombre/código de parroquia
(ADM3) — un dato que la sección 4 no estaba usando. La pregunta fue: ¿un
cruce a nivel de parroquia daría 100% tanto espacial como de nombres?

**Respuesta corta, verificada con los datos (no asumida):** no al 100% como
capa base — la cobertura de `parish_code` contra el ADM3 2024 de HDX es de
apenas ~80.7% en todo el dataset (79/80% en la mayoría de los años, sin una
tendencia clara por año — no es solo un problema de registros antiguos).
Adoptar parroquia como unidad base de análisis introduciría un vacío mayor
(19.3%) que el que tiene cantón hoy (0%). **Pero sí es extremadamente útil
como confirmación dirigida** de los casos ambiguos de la sección 4: cuando
`parish_code` existe en ADM3, su cantón "padre" (`ADM2_PCODE` de esa misma
parroquia) casi siempre coincide con el `canton_code` que SNGRE ya había
reportado — no con el cantón geométrico calculado en la sección 4.

In [ ]:
# Tabla de consulta: código de parroquia -> código de su cantón padre,
# tomada directamente de la propia capa ADM3 (ya trae ADM2_PCODE de cada
# parroquia). Esto es un segundo cruce por atributo (no espacial), igual de
# determinístico que el de la sección 2, solo que a un nivel más fino.
# Map from each parish to its canton
parish_to_canton = dict(zip(adm3["parish_code"], adm3["canton_code_from_parish"]))

# Para cada evento, ¿a qué cantón pertenece su parroquia reportada, según ADM3?
# NaN si el parish_code de SNGRE no existe en el ADM3 2024 (ver limitación
# de cobertura documentada arriba).
# Look up the canton implied by the reported parish
joined["canton_code_from_parish"] = joined["parish_code"].map(parish_to_canton)

# Bandera de confirmación: ¿el cantón derivado de la parroquia coincide con
# el cantón que SNGRE reportó directamente para el evento?
# Flag events whose reported canton agrees with the parish
joined["canton_confirmed_by_parish"] = joined["canton_code_from_parish"] == joined["canton_code"]

# Number of events with parish information
n_with_parish = joined["canton_code_from_parish"].notna().sum()
# Parish coverage as a percentage
pct_parish_coverage = n_with_parish / len(joined) * 100
# Log the parish coverage
logger.info(
    "Cobertura de parish_code contra ADM3 2024: %.1f%% (%d de %d eventos)",
    pct_parish_coverage, n_with_parish, len(joined),
)

# De los eventos CON parish_code válido, ¿confirma el cantón reportado?
# Share of events confirmed by the parish
confirm_rate = joined.loc[joined["canton_code_from_parish"].notna(), "canton_confirmed_by_parish"].mean()
# Log the confirmation rate
logger.info(
    "De los eventos con parroquia válida, confirman el cantón reportado por SNGRE: %.1f%%",
    confirm_rate * 100,
)

In [ ]:
# Aplicar la confirmación específicamente a los dos grupos ambiguos de la
# sección 4: (a) los 35 sin polígono cantonal, (b) los ~406 con cantón
# geométrico distinto del reportado. Para ambos, se cuenta cuántos casos
# quedan resueltos (la parroquia confirma el REPORTADO) vs. sin resolver
# (parroquia tampoco tiene código válido).
# Events outside every polygon
sin_poligono = joined[joined["canton_code_geo"].isna()]
# How many of those the parish confirms
resueltos_sin_poligono = sin_poligono["canton_confirmed_by_parish"].sum()
# Log how many were resolved
logger.info(
    "De los %d eventos sin polígono cantonal: %d quedan confirmados por parroquia (cantón reportado correcto)",
    len(sin_poligono), resueltos_sin_poligono,
)

# Events whose geometry disagrees with the report
geo_distinto = joined.loc[~joined["canton_geo_matches_reported"] & joined["canton_code_geo"].notna()]
# How many of those the parish confirms
resueltos_geo_distinto = geo_distinto["canton_confirmed_by_parish"].sum()
# Log how many were resolved
logger.info(
    "De los %d eventos con cantón geométrico distinto del reportado: %d quedan confirmados por parroquia",
    len(geo_distinto), resueltos_geo_distinto,
)

# Bandera final de confianza por evento, combinando las dos verificaciones
# independientes (geométrica y de parroquia) en una sola columna legible
# para fases posteriores — no colapsa a un solo número prematuramente
# (cada verificación queda también disponible por separado).
# Assign a confidence label to an event's canton
def confidence_label(row) -> str:
    """Clasifica la confianza en el cantón asignado a un evento.

    Args:
        row: fila con canton_geo_matches_reported y canton_confirmed_by_parish.

    Returns:
        'alta' si geometría o parroquia confirman el cantón reportado;
        'baja_sin_parroquia' si ninguna de las dos pudo confirmarlo por
        falta de dato (no es evidencia de error, es falta de evidencia);
        'revisar' si alguna verificación existe pero contradice al reportado
        sin que la otra lo confirme.

    English:
    Assign a confidence label to the canton of one event using the geometric and the parish checks.

    Args:
        row: Event row with the check columns.

    Returns:
        'alta' when either check confirms the canton, 'baja_sin_parroquia' when neither source is available, otherwise 'revisar'.
    """
    # Geometry agrees with the report
    geo_ok = row.get("canton_geo_matches_reported") is True
    # Parish agrees with the report
    parish_ok = row.get("canton_confirmed_by_parish") is True
    # Parish information is available
    parish_available = pd.notna(row.get("canton_code_from_parish"))
    # Either source confirms the canton
    if geo_ok or parish_ok:
        # Label for confirmed events
        return "alta"
    # No parish and no polygon: low confidence
    if not parish_available and pd.isna(row.get("canton_code_geo")):
        # Label for events with no second source
        return "baja_sin_parroquia"
    # Otherwise the event needs manual review
    return "revisar"

# Apply the label to every event
joined["canton_confidence"] = joined.apply(confidence_label, axis=1)
# Log the distribution of confidence labels
logger.info("Distribución de confianza final por evento: %s", joined["canton_confidence"].value_counts().to_dict())
# Show the distribution of confidence labels
joined["canton_confidence"].value_counts()

## 5. Armonización de nombres oficiales

Se añaden `province_name_official`/`canton_name_official` desde los límites
administrativos (fuente INEC), como complemento — no reemplazo — de los
nombres de texto libre que ya trae `hazard_event` desde SNGRE (que pueden
tener inconsistencias de tildes/mayúsculas entre registros).

**English:** 5. Harmonize official names

In [ ]:
# Seleccionar solo las columnas de interés del resultado del join espacial.
# Drop the helper column left by the spatial join
hazard_event_geo_final = joined.drop(columns=["index_right"], errors="ignore")

# Verificación rápida: cuántos registros distintos de 'province' (texto
# libre de SNGRE) mapean a un mismo province_name_official — cuantifica la
# inconsistencia de nombres que la armonización resuelve.
# Count name variants per official province
name_variants = hazard_event_geo_final.groupby("province_name_official")["province"].nunique()
# Log how many provinces have several spellings in the source
logger.info("Provincias con más de una variante de nombre en SNGRE: %d de %d", (name_variants > 1).sum(), len(name_variants))

## 6. Resumen de calidad de datos

**English:** 6. Data-quality summary

In [ ]:
# Dictionary with the quality summary
summary = {
    "total_eventos_geolocalizados": len(hazard_event_geo_final),
    "match_cantonal_geometrico_pct": round(match_rate * 100, 2),
    "eventos_sin_poligono_pct": round(n_no_polygon / len(joined) * 100, 2),
    "cobertura_parish_code_pct": round(pct_parish_coverage, 2),
    "confianza_alta_pct": round((hazard_event_geo_final["canton_confidence"] == "alta").mean() * 100, 2),
    "confianza_revisar_n": int((hazard_event_geo_final["canton_confidence"] == "revisar").sum()),
    "provincias_con_nombres_inconsistentes": int((name_variants > 1).sum()),
    "cantones_cubiertos_adm2": len(adm2),
    "parroquias_cubiertas_adm3": len(adm3),
}
# Print each summary item
for k, v in summary.items():
    # Print the key and its value
    print(f"{k}: {v}")
logger.info("Resumen Fase 2: %s", summary)

## 7. Exportación

Se exportan dos productos a `data/processed/`:

- `admin_boundaries.gpkg` (4 capas: país, provincia, cantón, parroquia) y sus
  versiones GeoJSON simplificadas de provincia/cantón para visualización web.
- `hazard_event_geo.gpkg`, ahora con las columnas de confirmación cruzada
  (`canton_code_from_parish`, `canton_confirmed_by_parish`, `canton_confidence`)
  añadidas en esta versión.

**English:** 7. Export

In [ ]:
# Directorio de salida (mismo que la Fase 1).
# Create the output folder if needed
out_dir = ensure_dir(processed_dir)

# DECISIÓN TÉCNICA: la carpeta del proyecto está montada como un sistema de
# archivos remoto (unidad de red/FUSE), que no soporta el bloqueo de
# archivos (file locking) que SQLite necesita para escribir un GeoPackage
# (formato basado en SQLite) — la escritura directa falla con
# "no such table: gpkg_contents". Se escribe primero en un directorio
# temporal LOCAL (tempfile), y luego se copian los archivos ya completos
# (copia simple de archivo sí funciona en la carpeta montada). Ver
# limitación documentada en docs/methodology/02_spatial_integration.md.
# Temporary folder utilities
import tempfile  # directorio temporal local, fuera del filesystem montado
# File copy utility
import shutil  # copiar archivos ya completos hacia out_dir

# Write to a temporary folder first to avoid partial files
with tempfile.TemporaryDirectory() as tmp:
    # Path of the temporary folder
    tmp_dir = Path(tmp)  # directorio local, con soporte SQLite completo

    # --- Capa de límites administrativos: un GeoPackage multi-capa ---
    # Path of the temporary GeoPackage
    gpkg_tmp = tmp_dir / config["admin_boundaries"]["output_gpkg"]
    # Write the country layer
    adm0.to_file(gpkg_tmp, layer="adm0_country", driver="GPKG")  # país (contexto)
    # Write the province layer
    adm1.to_file(gpkg_tmp, layer="adm1_province", driver="GPKG")  # provincia
    # Write the canton layer
    adm2.to_file(gpkg_tmp, layer="adm2_canton", driver="GPKG")  # cantón
    # Write the parish layer
    adm3.to_file(gpkg_tmp, layer="adm3_parish", driver="GPKG")  # parroquia

    # También GeoJSON de ADM1/ADM2 para previsualización web rápida (futuro
    # dashboard, Fase 0 §9). Se generalizan las geometrías (simplify con
    # tolerancia ~50 m, preservando topología) SOLO para estos GeoJSON: la
    # fuente trae vértices de altísima resolución (línea de costa detallada),
    # que produce GeoJSON de 92-293 MB — inviable para un mapa web. El GPKG
    # (sección anterior) conserva la geometría ORIGINAL sin simplificar, para
    # cualquier análisis espacial fino de fases futuras.
    # Simplification tolerance for web-friendly boundaries, in degrees
    SIMPLIFY_TOLERANCE_DEG = 0.0005  # ~50 m en el ecuador — documentado, no arbitrario silencioso
    # Copy the province layer
    adm1_simplified = adm1.copy()
    # Simplify province geometries
    adm1_simplified["geometry"] = adm1_simplified.geometry.simplify(SIMPLIFY_TOLERANCE_DEG, preserve_topology=True)
    # Copy the canton layer
    adm2_simplified = adm2.copy()
    # Simplify canton geometries
    adm2_simplified["geometry"] = adm2_simplified.geometry.simplify(SIMPLIFY_TOLERANCE_DEG, preserve_topology=True)

    # Write simplified provinces as GeoJSON
    adm1_simplified.to_file(tmp_dir / "adm1_province.geojson", driver="GeoJSON")
    # Write simplified cantons as GeoJSON
    adm2_simplified.to_file(tmp_dir / "adm2_canton.geojson", driver="GeoJSON")

    # --- Capa de eventos geolocalizados (incluye ahora la confirmación por parroquia) ---
    # Write the event points as a GeoPackage layer
    hazard_event_geo_final.to_file(tmp_dir / "hazard_event_geo.gpkg", layer="hazard_event", driver="GPKG")

    # Copiar todos los archivos ya completos y cerrados hacia out_dir (esto
    # sí es una operación simple de copia, soportada por el filesystem
    # montado — no requiere locking a nivel de SQLite).
    # List of files written to the final folder
    written_files = []
    # Loop over the temporary files
    for f in tmp_dir.iterdir():
        # Destination path in the processed folder
        dest = out_dir / f.name
        # Copy the file to its destination
        shutil.copy2(f, dest)
        # Remember the destination
        written_files.append(dest)
        # Log the file name and size
        logger.info("Copiado a destino final: %s (%.1f MB)", dest.name, dest.stat().st_size / 1e6)

# Print the output folder
print("Archivos escritos en:", out_dir)
# Loop over the written files
for f in written_files:
    # Print each file name
    print(" -", f.name)

## 8. Próximos pasos (Fase 3)

1. Análisis histórico descriptivo por provincia/cantón (conteos, tendencia)
   usando `hazard_event_geo.gpkg`, aprovechando la columna `canton_confidence`
   para ponderar o filtrar registros de baja confianza si el análisis lo requiere.
2. Revisar manualmente los eventos con `canton_confidence == "revisar"`
   (evidencia contradictoria entre geometría y parroquia) — grupo pequeño,
   candidato a verificación puntual contra el reporte original.
3. Contrastar el conteo de eventos por cantón contra el modelo de
   Escenarios/COE del workbook maestro (validación externa acordada).
4. Investigar los 5 eventos de la parroquia "5 De Agosto"/"Esmeraldas" (cantón
   Esmeraldas) cuyo parish_code no existe en el ADM3 2024 — únicos casos sin
   ninguna confirmación (ni geométrica ni de parroquia) tras esta fase.